# Notebook 04 of 7 — Events + Smart Money (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1440](https://github.com/prajoria/OpenBB/issues/1440) · Track A counterpart: [`../portfolio/04-events-and-smart-money.ipynb`](../portfolio/04-events-and-smart-money.ipynb).

---

## Where we are in Sam's story

NB03 gave me the static picture — what I own after ETF look-through and how concentrated it really is. Now the dynamic overlay: what's coming up on the calendar for these names in the next 30 days, and which are being bought or sold by people with better information than me. Free-only lane, so I'm relying on SEC filings (13F + Form 4) as the smart-money source and yfinance's recorded earnings calendar as the events source.

By the end we can answer:

> *Who else is trading these names right now, and what hits the calendar this week — using ONLY free-authoritative sources?*


## 0. Before we run anything

Same venv rule as every notebook in the series — `.venv_portfolio`. State goes into `.notebook_state/` (gitignored). We write to `smart_money_free.pkl` and `events_free.pkl` — do NOT overwrite Track A's `smart_money.pkl` / `events.pkl`.


In [ ]:
# [Track B / NB04 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB04 looks different from Track A

The smart-money core is actually *stronger* in the free-only lane than you might expect — 13F and Form 4 are both first-party SEC filings, so they're free-authoritative. Track A's `smart_money.rollup` router uses the same SEC feeds under the hood; the difference is that Track A adds a convenience composite (across 13F + insider + gov-trades) via `fmp_cached`-flavored code paths.

Where the tracks genuinely diverge:

1. **Earnings calendar** — there is no free-authoritative earnings-date    feed. Track A gets clean scheduled prints via `fmp_cached`. Track B    uses `yfinance.Ticker.earnings_dates` (Yahoo's recorded page — not    an exchange-authoritative feed but reliable for large-cap US names).
2. **Composite smart-money score** — Track A's rollup returns a single    number in [-1, +1] per name. Track B recreates the *shape* using a    fixture `SmartMoneyScoreItem` (labelled `example — signal shape as    of 2026-06-30` per STORY_BIBLE §3). The raw 13F + insider rows we    fetch below are live; the composite that combines them is illustrative    until a free-only rollup ships.
3. **News sentiment** — no free-authoritative source. Skipped honestly    with a pointer to Track A's news path.

Provider chain for this notebook:

| Data path | Provider (Track B) |
|---|---|
| Earnings calendar | `yfinance` recorded (Yahoo scraped, not exchange-auth) |
| Dividends history | `yfinance` recorded via `obb.equity.fundamental.dividends` |
| 13F institutional holdings | SEC EDGAR (`sec`) `equity.ownership.form_13f` |
| Insider transactions | SEC EDGAR (`sec`) `equity.ownership.insider_trading` |
| News + sentiment | *skipped — no free authoritative source* |
| Composite smart-money | Fixture (labelled) — mirrors Track A shape |

Bare-term pointer: Form 13F, Form 4, insider trading, STOCK Act, earnings announcement, ex-dividend date, reverse split, market sentiment, and event-driven were all cited with Investopedia links in Track A NB04. This notebook does not re-cite them.


## 1. Load the basket

Same 10-position basket — shared with Track A. Load `.notebook_state/basket.json` if present, else regenerate from the locked list. DO NOT modify basket.json (both tracks read it).


In [ ]:
# [Track B / NB04 §1] Load the shared 10-position basket (do NOT modify)
import json
from pathlib import Path

state = Path(".notebook_state")
basket_path = state / "basket.json"

BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12, "kind": "equity"},
    {"symbol": "NVDA",  "weight": 0.10, "kind": "equity"},
    {"symbol": "GOOGL", "weight": 0.08, "kind": "equity"},
    {"symbol": "AAPL",  "weight": 0.08, "kind": "equity"},
    {"symbol": "AMD",   "weight": 0.06, "kind": "equity"},
    {"symbol": "QQQ",   "weight": 0.15, "kind": "etf"},
    {"symbol": "VTI",   "weight": 0.20, "kind": "etf"},
    {"symbol": "VNQ",   "weight": 0.08, "kind": "etf"},
    {"symbol": "BND",   "weight": 0.10, "kind": "etf"},
    {"symbol": "GLD",   "weight": 0.03, "kind": "etf"},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path} (shared with Track A — not modified)")
else:
    basket = BASKET_LOCKED
    state.mkdir(exist_ok=True)
    basket_path.write_text(json.dumps(basket, indent=2), encoding="utf-8")
    print(f"Regenerated basket from locked list -> {basket_path}")

print(f"Positions: {len(basket)}")


Loaded basket from .notebook_state\basket.json (shared with Track A — not modified)
Positions: 10


## 2. Events calendar — yfinance recorded earnings per basket name

The calendar is not a prediction; it's a set of *scheduled reality checks*. Earnings prints, ex-dividend dates, and splits each change what a name is worth on a known date. The trader's problem isn't "will earnings beat" (nobody knows); it's "am I holding into a print I can't afford to be wrong about?"

Track A calls `obb.portfolio_intel.events.timeline` which unions earnings + dividends + splits from `fmp_cached`. Free-tier: we use `yfinance.Ticker.earnings_dates` per equity name (Yahoo scraped recorded page). It's not exchange-authoritative but it's the best free proxy for US mega-caps. Gap vs Track A: dividends + splits calendar isn't rolled into the same view here — they exist as historical series only, so we surface those separately in §2b.


In [ ]:
# [Track B / NB04 §2] Events calendar — yfinance recorded earnings
import warnings, time
warnings.filterwarnings("ignore")
import datetime as dt
import yfinance as yf  # last-resort free-tier events source (Yahoo scraped)

EQUITY_SYMBOLS = [p["symbol"] for p in basket if p.get("kind") == "equity"]
print(f"Fetching yfinance earnings_dates for {len(EQUITY_SYMBOLS)} equity names ...")

today = dt.date.today()
horizon = today + dt.timedelta(days=30)
timeline = []  # list of {symbol, event_date, event_type, meta}

t0 = time.perf_counter()
for sym in EQUITY_SYMBOLS:
    try:
        tk = yf.Ticker(sym)
        ed = tk.earnings_dates
    except Exception as exc:
        print(f"  {sym}: fetch failed — {type(exc).__name__}: {str(exc)[:80]}")
        continue
    if ed is None or ed.empty:
        continue
    for idx, row in ed.iterrows():
        ev_date = idx.date() if hasattr(idx, "date") else None
        if ev_date is None:
            continue
        if today <= ev_date <= horizon:
            timeline.append({
                "symbol": sym,
                "event_date": ev_date,
                "event_type": "earnings",
                "eps_estimate": float(row.get("EPS Estimate") or 0.0) or None,
            })
dt_s = time.perf_counter() - t0
print(f"  fetched in {dt_s:.1f}s — {len(timeline)} events in the next 30 days")

# Group by week
if timeline:
    timeline.sort(key=lambda x: x["event_date"])
    print()
    print("Upcoming earnings (next 30 days on your book):")
    print(f"  {'Date':<12}{'Symbol':<8}{'Event':<12}{'EPS est'}")
    print(f"  {'-'*12}{'-'*8}{'-'*12}{'-'*10}")
    for ev in timeline:
        est = f"{ev['eps_estimate']:.2f}" if ev['eps_estimate'] else "—"
        print(f"  {str(ev['event_date']):<12}{ev['symbol']:<8}{ev['event_type']:<12}{est}")
else:
    print("  (no earnings dates in the next 30 days for these names —")
    print("   could be a quiet stretch between earnings cycles.)")


Fetching yfinance earnings_dates for 5 equity names ...


  fetched in 7.6s — 3 events in the next 30 days

Upcoming earnings (next 30 days on your book):
  Date        Symbol  Event       EPS est
  ------------------------------------------
  2026-07-29  MSFT    earnings    4.24
  2026-07-30  AAPL    earnings    1.89
  2026-08-04  AMD     earnings    1.61


### 2b. Dividend history — free-tier via `obb.equity.fundamental.dividends`

The free-tier gap: no calendar of *forward* ex-dividend dates. But the historical dividend series is available via yfinance-recorded, and for a regular payer (MSFT pays quarterly, always mid-quarter), the next ex-date is a decent extrapolation of the last cadence. Track A gets explicit forward ex-dates from `fmp_cached`; Track B shows the last few payments and lets the reader eyeball the cadence.


In [ ]:
# [Track B / NB04 §2b] Dividend history — yfinance recorded, no forward calendar
from openbb import obb

div_summary = {}
for sym in ["MSFT", "AAPL"]:  # both regular quarterly payers
    try:
        r = obb.equity.fundamental.dividends(symbol=sym, provider="yfinance")
        rows = r.results or []
        recent = rows[-4:] if len(rows) >= 4 else rows
        div_summary[sym] = recent
    except Exception as exc:
        print(f"  {sym}: dividend fetch failed — {type(exc).__name__}")

for sym, rows in div_summary.items():
    print(f"{sym} — last {len(rows)} recorded dividends:")
    for row in rows:
        d = getattr(row, "ex_dividend_date", None)
        amt = getattr(row, "amount", None)
        print(f"  ex-date {d}    amount {amt}")
    print()
print("Gap vs Track A: no forward-calendar ex-dividend date here — free-tier")
print("has no authoritative forward dividend calendar. Track A gets scheduled")
print("ex-dates from the paid provider path; Track B reader must extrapolate cadence.")


MSFT — last 4 recorded dividends:
  ex-date 2025-08-21    amount 0.83
  ex-date 2025-11-20    amount 0.91
  ex-date 2026-02-19    amount 0.91
  ex-date 2026-05-21    amount 0.91

AAPL — last 4 recorded dividends:
  ex-date 2025-08-11    amount 0.26
  ex-date 2025-11-10    amount 0.26
  ex-date 2026-02-09    amount 0.26
  ex-date 2026-05-11    amount 0.27

Gap vs Track A: no forward-calendar ex-dividend date here — free-tier
has no authoritative forward dividend calendar. Track A gets scheduled
ex-dates from the paid provider path; Track B reader must extrapolate cadence.


## 3. Smart-money rollup — 13F path (SEC EDGAR)

Institutional holdings via Form 13F: quarterly filings from managers with $100M+ AUM, disclosing US long positions. 45-day filing lag; snapshot only (no intraquarter moves). Still the highest-signal retail-visible window into what large capital is actually doing.

Track B NB01 already discovered a gotcha: SEC 13F endpoints take a *manager CIK*, not a ticker. So we demo the shape using **Berkshire Hathaway** (CIK `0001067983`) since it's the canonical retail-visible 13F. To use this on your own manager watchlist, look up the CIK on EDGAR at `https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany` and swap it in.


In [ ]:
# [Track B / NB04 §3] Smart-money 13F path — SEC EDGAR (Berkshire example)
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

BERKSHIRE_CIK = "0001067983"  # canonical retail-visible 13F filer

try:
    r = obb.equity.ownership.form_13f(symbol=BERKSHIRE_CIK, provider="sec")
    rows_13f = r.results or []
except Exception as exc:
    print(f"form_13f err: {type(exc).__name__}: {str(exc)[:180]}")
    rows_13f = []

print(f"Berkshire Hathaway 13F rows: {len(rows_13f)}")
if rows_13f:
    # Sort by weight if present, else by value
    def _sort_key(x):
        w = getattr(x, "weight", None) or 0.0
        v = getattr(x, "value", None) or 0.0
        return -(w or v)
    top5 = sorted(rows_13f, key=_sort_key)[:5]
    period = getattr(top5[0], "period_ending", "?")
    print(f"Period ending: {period}")
    print()
    print(f"  {'Issuer':<28}{'Value ($)':>16}{'Weight':>10}")
    print(f"  {'-'*28}{'-'*16}{'-'*10}")
    for h in top5:
        issuer = str(getattr(h, "issuer", "?"))[:26]
        val = getattr(h, "value", None) or 0
        w = getattr(h, "weight", None)
        w_str = f"{w*100:.2f}%" if w else "—"
        print(f"  {issuer:<28}{val:>16,d}{w_str:>10}")
else:
    print("  (no 13F rows returned — SEC endpoint drift or CIK gap)")

print()
print("To use this on your own manager watchlist:")
print("  1. Look up the manager's CIK on EDGAR (search by name)")
print("  2. Swap BERKSHIRE_CIK above for the 10-digit CIK")
print("  3. Same code path returns their reported holdings")


Berkshire Hathaway 13F rows: 29
Period ending: 2026-03-31

  Issuer                             Value ($)    Weight
  ------------------------------------------------------
  APPLE INC                     57,843,260,493    21.99%
  AMERICAN EXPRESS CO           45,859,204,536    17.43%
  COCA COLA CO                  30,420,000,000    11.56%
  BANK AMERICA CORP             25,039,178,044     9.52%
  CHEVRON CORPORATION           17,457,364,606     6.64%

To use this on your own manager watchlist:
  1. Look up the manager's CIK on EDGAR (search by name)
  2. Swap BERKSHIRE_CIK above for the 10-digit CIK
  3. Same code path returns their reported holdings


## 4. Smart-money rollup — insider path (SEC Form 4)

The *fresh* smart-money stream: Form 4 filings, required within two business days of any insider transaction (officers, directors, 10%+ owners). 48-hour lag vs 45-day for 13F.

For the two most-held basket names (MSFT, NVDA), pull the recent Form 4 activity via SEC. Reading rule from Track A NB04 still applies: a *cluster* of insider buys (three or more insiders, same window, same direction) is the single highest-signal pattern in retail-visible data; a single insider sale is almost always noise (10b5-1 plans, tax lot management, options exercise).


In [ ]:
# [Track B / NB04 §4] Insider Form 4 via SEC — MSFT + NVDA
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

INSIDER_SYMBOLS = ["MSFT", "NVDA"]
insider_by_symbol = {}

for sym in INSIDER_SYMBOLS:
    try:
        r = obb.equity.ownership.insider_trading(symbol=sym, provider="sec", limit=10)
        rows = r.results or []
    except Exception as exc:
        print(f"  {sym}: fetch failed — {type(exc).__name__}: {str(exc)[:100]}")
        rows = []
    insider_by_symbol[sym] = rows
    print(f"{sym}: {len(rows)} recent insider filings")

print()
for sym, rows in insider_by_symbol.items():
    if not rows:
        continue
    print(f"--- {sym} — last {min(len(rows), 5)} Form 4 entries ---")
    print(f"  {'Filing':<12}{'Owner':<28}{'A/D':<5}{'Shares':>12}")
    for f in rows[:5]:
        d = getattr(f, "filing_date", "?")
        owner = str(getattr(f, "owner_name", "?"))[:26]
        ad = str(getattr(f, "acquisition_or_disposition", "?"))[0]
        shares = getattr(f, "securities_transacted", None) or 0
        try:
            shares_s = f"{float(shares):,.0f}"
        except Exception:
            shares_s = str(shares)
        print(f"  {str(d):<12}{owner:<28}{ad:<5}{shares_s:>12}")
    print()

# Cluster-buy detector — count distinct acquiring insiders per symbol
print("Cluster-buy scan (distinct acquiring insiders per symbol):")
for sym, rows in insider_by_symbol.items():
    buys = [f for f in rows if str(getattr(f, "acquisition_or_disposition", ""))[:1] == "A"]
    buyers = {getattr(f, "owner_name", "?") for f in buys}
    label = "CLUSTER" if len(buyers) >= 3 else "single/none"
    print(f"  {sym}: {len(buys)} acquisition filings from {len(buyers)} distinct insiders — {label}")



Found 10 total filings and 0 uncached entries to download, estimated download time: 0 seconds.



MSFT: 16 recent insider filings



Found 10 total filings and 0 uncached entries to download, estimated download time: 0 seconds.



NVDA: 10 recent insider filings

--- MSFT — last 5 Form 4 entries ---
  Filing      Owner                       A/D        Shares
  2026-07-15  Coleman Amy                 D              32
  2026-07-01  Hogan Kathleen T            N               0
  2026-06-16  Jolla Alice L.              A           5,004
  2026-06-15  Coleman Amy                 D              36
  2026-06-12  Walmsley Emma N             N               0

--- NVDA — last 5 Form 4 entries ---
  Filing      Owner                       A/D        Shares
  2026-07-06  COXE TENCH                  D         500,000
  2026-06-29  Shah Aarti S.               A           1,211
  2026-06-29  STEVENS MARK A              A           1,211
  2026-06-29  HUDSON DAWN E               A           1,211
  2026-06-29  SEAWELL A BROOKE            A           1,211

Cluster-buy scan (distinct acquiring insiders per symbol):
  MSFT: 7 acquisition filings from 7 distinct insiders — CLUSTER
  NVDA: 9 acquisition filings from 9 distinct i

## 5. News sentiment — skipped honestly

There is no free-authoritative news-sentiment feed. Yahoo publishes a news list per ticker but the tone/sentiment score is not part of that feed and requires an NLP layer we're not adding here. Track A uses `obb.news.company(provider="fmp")` for the raw feed and lets the reader eyeball headlines.

Track B: skip this overlay. When it agrees with smart-money it strengthens the case; when it disagrees, it's the interesting divergence. Both readings need a signal source Track B doesn't have. Route through Track A NB04 §6 if you have `fmp` access.


## 6. Composite smart-money reading — fixture per STORY_BIBLE §3

Track A's `smart_money.rollup` returns a single composite score in [-1, +1] per name, combining 13F drift + insider cluster + gov-trade ambient. That composite is what NB05 downstream consumes as `top_conviction`. A free-only composite that merges the raw 13F rows from §3 and Form 4 rows from §4 into a single score is out of scope for NB04 (belongs in a rollup service, not a demo cell).

So per STORY_BIBLE §3 item 3 we ship a *fixture* `SmartMoneyScoreItem` with the label `example — signal shape as of 2026-06-30`. NB05 Track B picks up the same shape as NB05 Track A and doesn't need to know the difference.

Reading rule I use (unchanged from Track A):

- **composite > +0.5 with insider > +0.7 AND form_13f > +0.5** —   high-conviction buy
- **composite > +0.3 with ONLY gov positive** — noise, ignore
- **composite < -0.3 with insider selling cluster + 13F drop** —   confirmed sell signal


In [ ]:
# [Track B / NB04 §6] Composite smart-money fixture (STORY_BIBLE §3 item 3)
# Labelled 'example — signal shape as of 2026-06-30' so no one mistakes
# these three rows for live signal. The raw 13F + insider fetches in
# §3 / §4 above ARE live; the composite that combines them is fixture.
from openbb_portfolio_intel.models import SmartMoneyScoreItem

fixture = [
    # example — signal shape as of 2026-06-30
    SmartMoneyScoreItem(
        symbol="MSFT",
        composite=+0.72,
        by_source={"insider": +0.85, "form_13f": +0.55, "gov": +0.05},
        signal_count=6,
    ),
    # example — signal shape as of 2026-06-30
    SmartMoneyScoreItem(
        symbol="NVDA",
        composite=+0.61,
        by_source={"insider": +0.30, "form_13f": +0.78, "gov": +0.10},
        signal_count=5,
    ),
    # example — signal shape as of 2026-06-30
    SmartMoneyScoreItem(
        symbol="AMD",
        composite=-0.58,
        by_source={"insider": -0.80, "form_13f": -0.35, "gov": 0.00},
        signal_count=4,
    ),
]

print("Composite smart-money [FIXTURE — example: signal shape as of 2026-06-30]:")
print(f"  {'Symbol':<8}{'Composite':>10}{'Signals':>10}  by_source")
print(f"  {'-'*8}{'-'*10}{'-'*10}  {'-'*40}")
for item in fixture:
    bs = ", ".join(f"{k}={v:+.2f}" for k, v in item.by_source.items())
    print(f"  {item.symbol:<8}{item.composite:>+10.2f}{item.signal_count:>10}  {bs}")

print()
print("Interpretation (Sam's read):")
print("  MSFT +0.72: insider cluster-buy on top of positive 13F drift. If NB03")
print("              said 'trim MSFT for concentration', this says wait — smarter")
print("              money is adding, not lightening.")
print("  NVDA +0.61: 13F leg carries this one. Institutions net-adding through")
print("              the window; insiders barely moved. Whales accumulating.")
print("  AMD  -0.58: coordinated insider selling + mild 13F trim. Closest thing")
print("              in this data set to a red flag.")


Composite smart-money [FIXTURE — example: signal shape as of 2026-06-30]:
  Symbol   Composite   Signals  by_source
  ----------------------------  ----------------------------------------
  MSFT         +0.72         6  insider=+0.85, form_13f=+0.55, gov=+0.05
  NVDA         +0.61         5  insider=+0.30, form_13f=+0.78, gov=+0.10
  AMD          -0.58         4  insider=-0.80, form_13f=-0.35, gov=+0.00

Interpretation (Sam's read):
  MSFT +0.72: insider cluster-buy on top of positive 13F drift. If NB03
              said 'trim MSFT for concentration', this says wait — smarter
              money is adding, not lightening.
  NVDA +0.61: 13F leg carries this one. Institutions net-adding through
              the window; insiders barely moved. Whales accumulating.
  AMD  -0.58: coordinated insider selling + mild 13F trim. Closest thing
              in this data set to a red flag.


## 7. Save state for NB05 (Track B)

Pickle the fetched artifacts to `.notebook_state/` for NB05 downstream. **Write to `smart_money_free.pkl` and `events_free.pkl`** — the no-`_free` names are owned by Track A and MUST NOT be overwritten.


In [ ]:
# [Track B / NB04 §7] Save Track-B artifacts — write ONLY to *_free.pkl paths
import pickle  # noqa: S403 — trusted local artifact under .notebook_state/
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

def _serialize_smi(item):
    return {
        "symbol": item.symbol,
        "composite": float(item.composite),
        "by_source": dict(item.by_source),
        "signal_count": int(item.signal_count),
    }

def _serialize_13f(row):
    return {
        "issuer": str(getattr(row, "issuer", "?")),
        "value": getattr(row, "value", None),
        "weight": getattr(row, "weight", None),
        "period_ending": str(getattr(row, "period_ending", "?")),
    }

def _serialize_insider(row):
    return {
        "filing_date": str(getattr(row, "filing_date", "?")),
        "owner_name": str(getattr(row, "owner_name", "?")),
        "acquisition_or_disposition": str(getattr(row, "acquisition_or_disposition", "?")),
        "securities_transacted": getattr(row, "securities_transacted", None),
    }

events_artifact = {
    "track": "B (free-only)",
    "basket": basket,
    "timeline": timeline,
    "source": "yfinance.Ticker.earnings_dates",
    "gap_vs_track_a": "no forward dividend/splits calendar in free tier",
}

smart_money_artifact = {
    "track": "B (free-only)",
    "basket": basket,
    "raw_13f_berkshire": [_serialize_13f(r) for r in rows_13f[:20]],
    "raw_insider_by_symbol": {
        sym: [_serialize_insider(r) for r in rows[:10]]
        for sym, rows in insider_by_symbol.items()
    },
    "top_conviction": [_serialize_smi(item) for item in fixture],
    "by_symbol": {item.symbol: _serialize_smi(item) for item in fixture},
    "is_fixture": True,
    "fixture_label": "example — signal shape as of 2026-06-30",
    "note": "Composite is fixture; raw 13F + insider rows are live SEC.",
}

events_out = state / "events_free.pkl"
sm_out = state / "smart_money_free.pkl"
events_out.write_bytes(pickle.dumps(events_artifact))
sm_out.write_bytes(pickle.dumps(smart_money_artifact))

print(f"Wrote (repo-rel):  {events_out}    {events_out.stat().st_size:,} bytes")
print(f"Wrote (repo-rel):  {sm_out}    {sm_out.stat().st_size:,} bytes")
print(f"Track A's events.pkl / smart_money.pkl NOT modified by this notebook.")


Wrote (repo-rel):  .notebook_state\events_free.pkl    976 bytes
Wrote (repo-rel):  .notebook_state\smart_money_free.pkl    3,706 bytes
Track A's events.pkl / smart_money.pkl NOT modified by this notebook.


---

## What is NOT in this notebook

Same gaps as Track A NB04 plus the free-tier delta:

- **Dark-pool prints.** No free feed. Same gap as Track A.
- **Short-interest changes as a distinct signal.** Currently rolled into   13F stream in Track A; free-tier has FINRA short-interest reports but   they're bi-monthly and not wired here.
- **Options-flow (unusual activity).** Same gap as Track A.
- **News sentiment aggregation.** No free-authoritative source. Track A   uses `fmp` for the raw news feed; Track B skips honestly (§5).
- **Analyst rating changes.** No free-authoritative source. Track A   covers this in NB08 via `fmp_cached`; no Track B analog planned.
- **Forward dividend / splits calendar.** Free-tier has no scheduled   ex-date feed; §2b shows historical dividends only and asks the reader   to extrapolate cadence.
- **Live composite smart-money score.** Track A's `smart_money.rollup`   merges the three streams into one number in [-1, +1]. Track B §6   ships a *fixture* labelled `example — signal shape as of 2026-06-30`   per STORY_BIBLE §3 item 3; a real free-only rollup would be its own   service, not a notebook cell.

## Preview of NB05 (Track B)

Three names look wrong. NB05 turns the signal work above into three specific candidate trades and diffs the portfolio to see what those trades would actually *do* to the basket. If the diff shows concentration going up instead of down, we don't place the trade.

## 📚 Further reading

All Investopedia links (earnings announcement, earnings surprise, ex-dividend date, reverse split, Form 13F, Form 4, insider trading, STOCK Act, market sentiment, event-driven) were cited in Track A NB04's Further Reading section — same vocabulary, same links. This notebook does not re-cite them.

**Canonical references** (unchanged from Track A):

- Ball, R. & Brown, P. — "An Empirical Evaluation of Accounting Income   Numbers," *Journal of Accounting Research* 6(2), 1968. The original   post-earnings-announcement drift paper.
- Cohen, L., Malloy, C. & Pomorski, L. — "Decoding Inside Information,"   *Journal of Finance* 67(3), 2012. The empirical case for weighting   routine vs. opportunistic insider trades differently.

**Free-authoritative sources used:**

- **SEC EDGAR Form 13F** — institutional holdings (§3)
- **SEC EDGAR Form 4** — insider transactions (§4)
- **yfinance recorded** — earnings dates + dividend history (§2, §2b)   — last-resort personal-use source, not exchange-authoritative
